# CNN Notebook
Intel Image Classification: training, scratch forward, and analysis.

In [ ]:
from pathlib import Path
import csv
import json
import os

import matplotlib.pyplot as plt
import numpy as np

from src.cnn.experiments import CLASS_NAMES, all_shared_experiments, get_experiment
from src.cnn.train_cnn import train_one
from src.cnn import compare_shared_non_shared, evaluate_cnn, plotting, scratch_compare, visualize_features
from src.dataset.prepare_intel_dataset import ensure_prepared_dataset
from src.utils.image_utils import list_image_paths, load_images

## 0. Setup & Paths

In [ ]:
repo_root = Path(".").resolve()
is_kaggle = Path("/kaggle/input").exists()

intel_source_root = Path(
    os.getenv(
        "INTEL_DATA_ROOT",
        "/kaggle/input/intel-image-classification" if is_kaggle else "src/dataset",
    )
 )
prepared_root = Path(
    os.getenv(
        "INTEL_PREPARED_ROOT",
        "/kaggle/working/intel_prepared" if is_kaggle else "src/dataset",
    )
 )

cnn_output_dir = Path(
    os.getenv("CNN_OUTPUT_DIR", "/kaggle/working/outputs/cnn" if is_kaggle else "outputs/cnn")
 )
cnn_output_dir.mkdir(parents=True, exist_ok=True)

cnn_image_size = 64
cnn_experiment_id = "d1_f16_k3_max"
cnn_non_shared = False
cnn_epochs = 1
cnn_batch_size = 16

run_prepare = False
run_train_single = False
run_train_all = False
run_non_shared = False

## 1. Dataset preparation (Intel layout to train/val/test)

In [ ]:
if run_prepare:
    cnn_data_root = ensure_prepared_dataset(
        data_root=intel_source_root,
        prepared_root=prepared_root,
        val_fraction=0.15,
        overwrite=False,
    )
elif (prepared_root / "train").exists():
    cnn_data_root = prepared_root
else:
    cnn_data_root = intel_source_root

cnn_data_root

## 2. Sanity check loader

In [ ]:
if (cnn_data_root / "train").exists():
    paths, labels, classes = list_image_paths(cnn_data_root / "train", class_names=CLASS_NAMES)
    sample_paths = paths[:16]
    sample_images = load_images(sample_paths, image_size=(cnn_image_size, cnn_image_size), normalize=True)
    (sample_images.shape, labels[:16].shape, classes)
else:
    print("Dataset CNN belum siap. Cek cnn_data_root.")

## 3. Experiment grid (16 configs)

In [ ]:
configs = all_shared_experiments(image_size=cnn_image_size)
config_rows = [cfg.to_dict() for cfg in configs]
config_rows[:4]

## 4. Train single experiment

In [ ]:
if run_train_single and (cnn_data_root / "train").exists():
    config = get_experiment(cnn_experiment_id, image_size=cnn_image_size)
    exp_dir = train_one(
        config=config,
        data_root=cnn_data_root,
        output_dir=cnn_output_dir,
        epochs=cnn_epochs,
        batch_size=cnn_batch_size,
        non_shared=cnn_non_shared,
    )
    exp_dir
else:
    print("run_train_single=False atau dataset belum siap.")

## 5. Train all shared experiments (optional)

In [ ]:
if run_train_all and (cnn_data_root / "train").exists():
    for config in all_shared_experiments(image_size=cnn_image_size):
        train_one(
            config=config,
            data_root=cnn_data_root,
            output_dir=cnn_output_dir,
            epochs=cnn_epochs,
            batch_size=cnn_batch_size,
            non_shared=False,
        )
    print("Selesai run_all shared experiments")
else:
    print("run_train_all=False atau dataset belum tersedia.")

## 6. Summarize experiments + pick best

In [ ]:
summary_rows = plotting.summarize_experiments(cnn_output_dir, cnn_output_dir / "summary.csv")
best_row = summary_rows[0] if summary_rows else None
best_row

## 7. Evaluate best + scratch compare

In [ ]:
eval_metrics = None
scratch_metrics = None
if best_row and (cnn_data_root / "test").exists():
    model_dir = cnn_output_dir / best_row["experiment"]
    model_path = model_dir / "model.keras"
    if model_path.exists():
        eval_metrics = evaluate_cnn.evaluate_model(
            model_path=model_path,
            data_root=cnn_data_root,
            output_dir=cnn_output_dir / "eval",
            image_size=cnn_image_size,
            batch_size=cnn_batch_size,
        )
        scratch_metrics = scratch_compare.compare_keras_scratch(
            model_path=model_path,
            split_root=cnn_data_root / "test",
            output_path=cnn_output_dir / "scratch_compare.json",
            image_size=cnn_image_size,
            max_samples=30,
        )
    else:
        print("Model terbaik belum ada di", model_dir)
else:
    print("Belum ada summary atau dataset test belum siap.")

eval_metrics, scratch_metrics

## 8. Shared vs non-shared comparison

In [ ]:
def _strip_prefix(name: str) -> str:
    for prefix in ("shared_", "non_shared_"):
        if name.startswith(prefix):
            return name[len(prefix):]
    return name

comparison = None
if best_row:
    exp_id = _strip_prefix(best_row["experiment"])
    shared_dir = cnn_output_dir / f"shared_{exp_id}"
    non_shared_dir = cnn_output_dir / f"non_shared_{exp_id}"
    if run_non_shared and (cnn_data_root / "train").exists():
        config = get_experiment(exp_id, image_size=cnn_image_size)
        train_one(
            config=config,
            data_root=cnn_data_root,
            output_dir=cnn_output_dir,
            epochs=cnn_epochs,
            batch_size=cnn_batch_size,
            non_shared=True,
        )
    if shared_dir.exists() and non_shared_dir.exists():
        comparison = compare_shared_non_shared.compare(
            shared_dir=shared_dir,
            non_shared_dir=non_shared_dir,
            output_path=cnn_output_dir / "shared_vs_non_shared.csv",
        )
    else:
        print("Shared/non-shared artifacts belum lengkap.")
else:
    print("Best row belum tersedia.")

comparison

## 9. Loss curves + summary table

In [ ]:
if summary_rows:
    top5 = summary_rows[:5]
    top5
else:
    print("Belum ada summary. Jalankan training/summarize dulu.")

loss_examples = []
if summary_rows:
    for row in summary_rows[:3]:
        loss_path = cnn_output_dir / row["experiment"] / "loss.png"
        if loss_path.exists():
            loss_examples.append(loss_path)
loss_examples

## 10. Bonus: feature maps + Grad-CAM

In [ ]:
viz_image_path = Path("src/dataset/test/buildings/0001.jpg")
viz_layer_name = None  # isi nama layer conv jika mau spesifik
if best_row:
    model_path = cnn_output_dir / best_row["experiment"] / "model.keras"
    if model_path.exists() and viz_image_path.exists():
        visualize_features.visualize(
            model_path=model_path,
            image_path=viz_image_path,
            output_dir=cnn_output_dir / "visuals",
            image_size=cnn_image_size,
            layer_name=viz_layer_name,
            class_index=None,
            max_channels=32,
            alpha=0.4,
        )
        print("Saved visuals to", cnn_output_dir / "visuals")
    else:
        print("Model atau image belum ada.")
else:
    print("Best row belum tersedia.")

## 11. Bonus: batch inference sanity (scratch)

In [ ]:
from src.cnn.cnn_scratch.layers import Conv2D, Dense, Flatten, LocallyConnected2D, MaxPooling2D
from src.cnn.cnn_scratch.model import SequentialScratchModel

def check_shared_batch() -> None:
    rng = np.random.default_rng(0)
    x = rng.normal(size=(4, 8, 8, 3)).astype(np.float32)
    kernel = rng.normal(size=(3, 3, 3, 4)).astype(np.float32)
    bias = np.zeros((4,), dtype=np.float32)

    conv = Conv2D(kernel=kernel, bias=bias, activation="relu")
    pool = MaxPooling2D(pool_size=(2, 2))
    flat = Flatten()
    dense = Dense(rng.normal(size=(3 * 3 * 4, 5)).astype(np.float32), np.zeros((5,), dtype=np.float32))

    model = SequentialScratchModel([conv, pool, flat, dense])
    out = model.predict(x)
    assert out.shape == (4, 5), out.shape
    print("Shared batch output shape:", out.shape)

def check_non_shared_batch() -> None:
    rng = np.random.default_rng(1)
    x = rng.normal(size=(3, 8, 8, 3)).astype(np.float32)
    kh, kw, cin, cout = 3, 3, 3, 2
    out_h = (8 - kh) + 1
    out_w = (8 - kw) + 1
    positions = out_h * out_w
    kernel = rng.normal(size=(positions, kh * kw * cin, cout)).astype(np.float32)
    bias = np.zeros((positions, cout), dtype=np.float32)

    local = LocallyConnected2D(kernel=kernel, bias=bias, kernel_size=(kh, kw), activation="relu")
    out = local.forward(x)
    assert out.shape == (3, out_h, out_w, cout), out.shape
    print("Non-shared batch output shape:", out.shape)

check_shared_batch()
check_non_shared_batch()

## 12. Bonus: backward propagation sanity (scratch)

In [ ]:
from src.cnn.cnn_scratch.layers import Conv2D as ScratchConv2D, Dense as ScratchDense
from src.cnn.cnn_scratch.losses import softmax_cross_entropy_loss

def _relative_error(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    denom = np.maximum(eps, np.maximum(np.abs(a), np.abs(b)))
    return float(np.max(np.abs(a - b) / denom))

def _dense_loss(x: np.ndarray, kernel: np.ndarray, bias: np.ndarray) -> float:
    out = x @ kernel + bias
    return float(np.sum(out))

def check_dense_backward() -> None:
    rng = np.random.default_rng(0)
    x = rng.normal(size=(2, 4)).astype(np.float32)
    kernel = rng.normal(size=(4, 3)).astype(np.float32)
    bias = rng.normal(size=(3,)).astype(np.float32)

    layer = ScratchDense(kernel.copy(), bias.copy(), activation=None)
    out = layer.forward(x)
    upstream = np.ones_like(out)
    layer.backward(upstream)

    eps = 1e-4
    grad_num = np.zeros_like(kernel)
    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            orig = kernel[i, j]
            kernel[i, j] = orig + eps
            loss_pos = _dense_loss(x, kernel, bias)
            kernel[i, j] = orig - eps
            loss_neg = _dense_loss(x, kernel, bias)
            grad_num[i, j] = (loss_pos - loss_neg) / (2 * eps)
            kernel[i, j] = orig

    err = _relative_error(layer.grad_kernel, grad_num)
    print("Dense grad kernel rel err:", err)

def _conv_loss(x: np.ndarray, kernel: np.ndarray) -> float:
    layer = ScratchConv2D(kernel=kernel, bias=None, strides=(1, 1), padding="valid", activation=None)
    out = layer.forward(x)
    return float(np.sum(out))

def check_conv2d_backward() -> None:
    rng = np.random.default_rng(1)
    x = rng.normal(size=(1, 4, 4, 1)).astype(np.float32)
    kernel = rng.normal(size=(3, 3, 1, 1)).astype(np.float32)

    layer = ScratchConv2D(kernel=kernel.copy(), bias=None, strides=(1, 1), padding="valid", activation=None)
    out = layer.forward(x)
    upstream = np.ones_like(out)
    layer.backward(upstream)

    eps = 1e-4
    grad_num = np.zeros_like(kernel)
    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            for c in range(kernel.shape[2]):
                for k in range(kernel.shape[3]):
                    orig = kernel[i, j, c, k]
                    kernel[i, j, c, k] = orig + eps
                    loss_pos = _conv_loss(x, kernel)
                    kernel[i, j, c, k] = orig - eps
                    loss_neg = _conv_loss(x, kernel)
                    grad_num[i, j, c, k] = (loss_pos - loss_neg) / (2 * eps)
                    kernel[i, j, c, k] = orig

    err = _relative_error(layer.grad_kernel, grad_num)
    print("Conv2D grad kernel rel err:", err)

check_dense_backward()
check_conv2d_backward()
logits = np.array([[1.0, -1.0, 0.5]], dtype=np.float32)
labels = np.array([2], dtype=np.int64)
loss, grad = softmax_cross_entropy_loss(logits, labels)
print("Softmax CE loss:", loss, "grad shape:", grad.shape)

## 13. Bonus: forward sanity (scratch)

In [ ]:
from src.cnn.cnn_scratch.activations import softmax
from src.cnn.cnn_scratch.layers import Conv2D as ScratchConv2D2, Dense as ScratchDense2, Flatten as ScratchFlatten, MaxPooling2D as ScratchMaxPool

x = np.arange(1 * 4 * 4 * 1, dtype=np.float32).reshape((1, 4, 4, 1))
kernel = np.ones((3, 3, 1, 2), dtype=np.float32)
bias = np.array([0.0, 1.0], dtype=np.float32)
conv = ScratchConv2D2(kernel, bias=bias, activation="relu")
conv_out = conv.forward(x)
assert conv_out.shape == (1, 2, 2, 2), conv_out.shape

pool = ScratchMaxPool(pool_size=(2, 2))
pool_out = pool.forward(x)
assert pool_out.tolist() == [[[[5.0], [7.0]], [[13.0], [15.0]]]], pool_out

flat = ScratchFlatten().forward(x)
assert flat.shape == (1, 16)
assert flat[0, 0] == 0 and flat[0, -1] == 15

dense = ScratchDense2(np.ones((16, 3), dtype=np.float32), np.zeros((3,), dtype=np.float32), activation="softmax")
probs = dense.forward(flat)
assert probs.shape == (1, 3)
assert np.allclose(np.sum(probs, axis=1), 1.0)
assert np.allclose(np.sum(softmax(np.array([[1.0, 2.0, 3.0]])), axis=1), 1.0)
print("scratch sanity checks passed")